data description: https://www.kaggle.com/competitions/rental-price-predict/data

- merges listings + reviews
- cleans HTML, filters English, filters ≥3 sentences, cleans price, keep not na of price, neighborhood
- outputs an embedding-ready dataset

In [3]:
import pandas as pd
from pathlib import Path

LISTINGS_CSV = "../../airbnb-broward-data/listings.csv"
REVIEWS_CSV = "../../airbnb-broward-data/reviews.csv"

listings = pd.read_csv(LISTINGS_CSV, low_memory=False)
reviews = pd.read_csv(REVIEWS_CSV, low_memory=False)

print('\n--- Listings ---')
print('shape:', listings.shape)

print('\n--- Listings head ---')
print(listings.head().to_string())

print('\n\n--- Reviews ---')
print('shape:', reviews.shape)

print('\n--- Reviews head ---')
print(reviews.head().to_string())



--- Listings ---
shape: (16822, 79)

--- Listings head ---
       id                          listing_url       scrape_id last_scraped           source                                              name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  description                                                                                                                                                                                                                                                                                                                                 

In [4]:
listings.columns

Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'neighborhood_overview', 'picture_url', 'host_id',
       'host_url', 'host_name', 'host_since', 'host_location', 'host_about',
       'host_response_time', 'host_response_rate', 'host_acceptance_rate',
       'host_is_superhost', 'host_thumbnail_url', 'host_picture_url',
       'host_neighbourhood', 'host_listings_count',
       'host_total_listings_count', 'host_verifications',
       'host_has_profile_pic', 'host_identity_verified', 'neighbourhood',
       'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude',
       'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
       'minimum_nights', 'maximum_nights', 'minimum_minimum_nights',
       'maximum_minimum_nights', 'minimum_maximum_nights',
       'maximum_maximum_nights', 'minimum_nights_avg_ntm',
       'maximum_nights_avg_ntm', 'ca

In [5]:
reviews.columns

Index(['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments'], dtype='object')

In [6]:
listings["description"] = (
    listings["description"]
    .astype(str)
    .str.replace(r"[\r\n]+", " ", regex=True)   # remove real newlines
    .str.replace(r"<br\s*/?>", " ", regex=True) # remove <br>, <br/>, <br />
)

reviews["comments"] = (
    reviews["comments"]
    .astype(str)
    .str.replace(r"[\r\n]+", " ", regex=True)
    .str.replace(r"<br\s*/?>", " ", regex=True) # remove <br>, <br/>, <br /> 
    .str.replace(r"[^\w\s,.!?']", "", regex=True)
)

merged = listings.merge(
    reviews,
    left_on="id",
    right_on="listing_id"
)

print(merged.head().to_string())

    id_x                         listing_url       scrape_id last_scraped           source                                              name                                                                                                                                                                                                                                                                                                                                                                                                                                                                        description                                                                                                                                                                                                                                                                                                                                                                                                         

In [7]:
print('\n\n--- Merged ---')
print('shape:', merged.shape)

print('\n--- Merged head ---')
print(merged.head().to_string())



--- Merged ---
shape: (649616, 85)

--- Merged head ---
    id_x                         listing_url       scrape_id last_scraped           source                                              name                                                                                                                                                                                                                                                                                                                                                                                                                                                                        description                                                                                                                                                                                                                                                                                                                                               

In [8]:
small = merged[[
    "id_x",
    "id_y",
    "name",
    "description",
    "comments",
    "review_scores_rating", 'review_scores_accuracy',
       'review_scores_cleanliness', 'review_scores_checkin',
       'review_scores_communication', 'review_scores_location',
       'review_scores_value',
    'neighbourhood_cleansed',
    'price',
    'availability_30', 'availability_60', 'availability_90',
       'availability_365', 'availability_eoy'
]].rename(columns={
    "id_x": "listing_id",
    "id_y": "review_id",
    "name": "listing_name",
    "description": "description",
    "comments": "review",
    "review_scores_rating": "rating",
    "neighbourhood_cleansed": "neighborhood"
})
print('\n\n--- Small ---')
print('shape:', small.shape)
print('\n--- Small head ---')
print(small.head().to_string())



--- Small ---
shape: (649616, 19)

--- Small head ---
   listing_id  review_id                                      listing_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                        description                                                                                                                                                                                                                                                                                                                                                                                                                   

In [9]:
import re

def sentence_count(text):
    if not isinstance(text, str):
        return 0
    # split on ., !, or ?
    sentences = re.split(r'[.!?]+', text)
    # remove empty fragments
    sentences = [s.strip() for s in sentences if len(s.strip()) > 0]
    return len(sentences)

In [10]:
import langid

def is_english(text, min_chars=20):
    if not isinstance(text, str) or len(text.strip()) < min_chars:
        return False
    lang, score = langid.classify(text)
    return lang == "en"

In [11]:
def clean_price(x):
    return float(str(x).replace("$", "").replace(",", "").strip())


In [12]:
small["desc_sentences"] = small["description"].apply(sentence_count)
small["review_sentences"] = small["review"].apply(sentence_count)
small["is_english"] = small["review"].apply(is_english)
small['price'] = small['price'].apply(clean_price)
filtered = small[
    (small["desc_sentences"] >= 3) &
    (small["review_sentences"] >= 3) &
    (small["is_english"]) &
    (small["price"].notna()) &
    (small["neighborhood"].notna())
]

In [13]:
filtered = filtered.drop(columns=["desc_sentences", "review_sentences", "is_english"])
filtered.shape

(251215, 19)

In [51]:
filtered.head()

,listing_id,review_id,listing_name,description,review,rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,neighborhood,price,availability_30,availability_60,availability_90,availability_365,availability_eoy
1,27886,2359368,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,"What can I say, Flip provided the perfect Amst...",4.92,4.9,4.94,4.95,4.93,4.9,4.78,Centrum-West,132.0,2,5,16,17,17
3,27886,7509410,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,Just a superb way to spend time in Amsterdam. ...,4.92,4.9,4.94,4.95,4.93,4.9,4.78,Centrum-West,132.0,2,5,16,17,17
5,27886,21170224,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,Why stay anywhere else in the city of canals w...,4.92,4.9,4.94,4.95,4.93,4.9,4.78,Centrum-West,132.0,2,5,16,17,17
7,27886,21700505,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,"The place was beautiful and bright, even when ...",4.92,4.9,4.94,4.95,4.93,4.9,4.78,Centrum-West,132.0,2,5,16,17,17
8,27886,21848508,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,Flip's houseboat is very cool. It is tastefull...,4.92,4.9,4.94,4.95,4.93,4.9,4.78,Centrum-West,132.0,2,5,16,17,17


In [53]:
filtered.columns

Index(['listing_id', 'review_id', 'listing_name', 'description', 'review',
       'rating', 'review_scores_accuracy', 'review_scores_cleanliness',
       'review_scores_checkin', 'review_scores_communication',
       'review_scores_location', 'review_scores_value', 'neighborhood',
       'price', 'availability_30', 'availability_60', 'availability_90',
       'availability_365', 'availability_eoy'],
      dtype='object')

In [52]:
filtered.to_csv("filtered_listings_reviews.csv", index=False)